### 42.接雨水
给定 n 个非负整数表示每个宽度为 1 的柱子的高度图，计算按此排列的柱子，下雨之后能接多少雨水。

示例 1：

输入：height = [0,1,0,2,1,0,1,3,2,1,2,1]
输出：6

解释：上面是由数组 [0,1,0,2,1,0,1,3,2,1,2,1] 表示的高度图，在这种情况下，可以接 6 个单位的雨水（蓝色部分表示雨水）。

示例 2：

输入：height = [4,2,0,3,2,5]
输出：9

#### 1.前后缀分解
由于数组每个位置数表示宽度为 1，故将每个位置视为池子。
每个单位池子位置可容纳雨水量：
1. 左右边界 由 此位置 左子数组 和 右子数组 的最大值决定。（此位置前后缀分离）
2. 此单位池子 **最大值** 取左右子数组最大值中较小的一个。（底座宽度为1，容积即为高） 

**步骤**
1. 创建左右边界数组，记录每个位置 前后缀子串 边界最大值。
   1. left 存储 `[0,i]`位置的最大值
   2. right 存储 `[i,n-1]`位置的最大值 
2. 每个位置容积：由 较小左右边界 - 当前位置高 决定
   1. h = min(left[i], right[i]) - height[i]
   2. 累加 h  
如示例1：对应 前后缀数组
- height = [0,1,0,2,1,0,1,3,2,1,2,1]
- pre_max = [0,1,1,2,2,2,2,3,3,3,3,3]
- post_max = [3,3,3,3,3,3,3,3,3,3,2,1]

时间复杂度：O(n)  **时间复杂度已最优**
空间复杂度：O(n)


In [3]:
from typing import List
class Solution:
    def trap(self, height: List[int]) -> int:
        # 用前后缀数组记录，当前位置左侧最高边界 和 右侧最高边界
        # 前缀数组 从前向后取max 非递减
        # 后缀数组 从后向前取max 非递增
        
        n = len(height)
        pre_max = [0] * n
        pre_max[0] = height[0]
        for i in range(1, n):
            pre_max[i] = max(pre_max[i-1], height[i]) # 非递减
        
        post_max = [0] * n
        post_max[-1] = height[-1]
        for i in range(n-2, -1, -1): # 倒着算
            post_max[i] = max(post_max[i+1], height[i]) # 非递增
        
        ans = 0
        for h, pre, post in zip(height, pre_max, post_max):
            ans += min(pre, post) - h

        return ans

height = [0,1,0,2,1,0,1,3,2,1,2,1]
print(Solution().trap(height))

6


#### 2. 相向双指针
在「谁小移动谁」的规则下，相遇的位置一定是最高的柱子，这个柱子是无法接水的。

**双向指针：实际是将 前后缀数组 空间 优化 为指针O(1)**
思路：
1. 确定两个指针，指向开头结尾，分别记录其当前位置的 最大前缀高 和 最大后缀高。
2. 移动两个指针，直到两个指针相遇。
    1. 在位置 i 时，此时左右指针未相遇，中间存在未遍历区域，如何确认 i 位置处左右最高边界？
    2. 当 左指针遍历 left 位置时，若 left 的左最高边界 pre_max < 此刻 right记录的最高边界，由取最小原则，此处 i 的最高边界 即为 pre_max。
    3. 反之，后缀右指针遍历到 位置 right 时，若 right 的右最高边界 post_max < 此刻 left 记录的最高边界，由取最小原则，此处 right 的最高边界 即为 post_max。

In [4]:
class Solution:
    def trap(self, height: List[int]) -> int:
        # 用前后缀数组记录，当前位置左侧最高边界 和 右侧最高边界
        # 将前后缀数组替换为 最高单位值，并由左右指针遍历 实时计算
        # 每次只 累计 前后指针中 小的一侧 的容积，移动小的一侧指针

        n = len(height)
        ans = 0
        left, right = 0, n -1
        pre_max, post_max = 0, 0

        while left <= right: # 相遇位置，由于位置有宽度 1 ，可计算容积
            pre_max = max(pre_max, height[left])
            post_max = max(post_max,height[right])

            # 判断此时 前缀小还是后缀小，计算 小的一侧 指针位置容积
            if pre_max < post_max:
                ans += pre_max - height[left]
                left += 1
            else:
                ans += post_max -height[right]
                right -= 1

        return ans
    
height = [0,1,0,2,1,0,1,3,2,1,2,1]
print(Solution().trap(height))

6


#### 3.单调栈
上面的方法相当于「竖着」计算面积，单调栈的做法相当于「横着」计算面积。

这个方法可以总结成 16 个字：找上一个更大元素，在找的过程中填坑。

注意 while 中加了等号，这可以让栈中没有重复元素，从而在有很多重复元素的情况下，使用更少的空间。

作者：灵茶山艾府
链接：https://leetcode.cn/problems/trapping-rain-water/solutions/

In [ ]:
class Solution:
    def trap(self, height: List[int]) -> int:
        ans = 0
        st = [] # 单调栈，找上一个最大的元素坐标
        for i, h in enumerate(height):
            while st and height[st[-1]] <= h: # 当前元素比栈顶元素大，说明栈顶元素被包围了
                bottom_h = height[st.pop()] # 栈顶元素出栈，记录被包围的元素高度
                if not st: # 栈为空，没有栈顶元素
                    break
                left = st[-1]
                dh = min(height[left], h) - bottom_h # 计算被包围的元素高度差
                ans += (i - left - 1) * dh
            
            st.append(i) # 当前元素入栈
        return ans
